# Contexto documental e detecção supervisionada

Nova hipótese de dados com avaliação pareada e degradação de contexto.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

root = Path.cwd()
while not (root / "ml/config.py").exists():
    if root.parent == root:
        raise RuntimeError("Raiz do projeto não encontrada")
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

## Extensão com contexto documental anterior à transação

Nova hipótese sintética de informação, não uma correção retroativa dos dados antigos. Os modelos e a comparação sem contexto usam os mesmos lotes. Quantidade e preço documental são emitidos antes do lançamento e antes da adulteração, com ruído, ausência e erros independentes do rótulo.

In [2]:
from ml.context_run import DEST as CONTEXT_RESULTS
from ml.context_report import make_figures as context_figures
from ml.data import sha256 as context_sha256
from ml.config import ROOT as CONTEXT_ROOT

context_protocol = json.loads(
    (CONTEXT_RESULTS / "protocolo.json").read_text(encoding="utf-8")
)
for filename, expected in context_protocol["metodos_sha256"].items():
    assert context_sha256(CONTEXT_ROOT / filename) == expected, filename
display(
    Markdown((CONTEXT_RESULTS / "RESULTADOS_CONTEXTO.md").read_text(encoding="utf-8"))
)
display(pd.read_csv(CONTEXT_RESULTS / "validacao.csv").round(4))
display(pd.read_csv(CONTEXT_RESULTS / "teste_agregado.csv").round(4))
display(pd.read_csv(CONTEXT_RESULTS / "segmentos.csv").round(4))
context_plots = context_figures()
if "show_figures" in globals():
    show_figures(context_plots, "6.6. Extensão documental")
else:
    for figure in context_plots:
        display(figure)
        plt.close(figure)
if "required" in globals():
    required.extend(
        CONTEXT_RESULTS / name
        for name in [
            "protocolo.json",
            "selecao_congelada.json",
            "resumo.json",
            "validacao.csv",
            "teste_agregado.csv",
            "segmentos.csv",
            "RESULTADOS_CONTEXTO.md",
        ]
    )

# Extensão supervisionada com contexto documental

## O que foi alterado

O histórico de valores tem capacidade limitada para distinguir uma operação legítima incomum de uma adulteração. Esta rodada acrescenta uma hipótese de informação: existe um pedido com quantidade e preço, emitido entre um e cinco dias antes da transação. É uma construção sintética proposta pelo projeto, não documentação empresarial observada.

O gerador foi refeito em duas etapas causais. Primeiro cria a operação comercial, picos legítimos, mudanças de quantidade, documento e faturamento normal. Depois um fluxo aleatório independente injeta adulteração somente no valor da transação. Os campos de documento não recebem `flag_anomalia`, fatores de injeção ou o valor do gabarito. A referência documental resulta da quantidade vezes o preço informados no pedido. Ela não é copiada do `valor_legitimo` dos experimentos anteriores.

Os documentos têm ruído logarítmico de 0,03 no preço, ausência esperada de 15% e probabilidade de erro adicional de 1,5%. O faturamento normal tem ruído logarítmico de 0,045. Essas intensidades são hipóteses experimentais, não estimativas de uma empresa. A taxa de adulteração permanece aproximadamente 4%, com metade de alterações sutis. Documentos ruidosos ou ausentes ocorrem independentemente da adulteração. Isso torna a nova tarefa uma detecção de inconsistência entre duas medições, além de raridade histórica.

## Treino, validação e teste

- Treino supervisionado: seis clientes, sementes120001–120006. Transações de março a junho/2026, com históricos desde janeiro. Foram 39300 exemplos, incluindo 1565 adulterações rotuladas pelo simulador. Nenhum rótulo é apresentado como investigação humana real.
- Validação: três outros clientes, sementes220001–220003, lote julho/2026 e históricos anteriores. Há 5090 exemplos. Ajuste de limiar e escolha de candidato somente aqui.
- Teste reservado: cinco outros clientes, sementes320001–320005, lote setembro/2026 e históricos janeiro–agosto. Escolha e modelos foram congelados antes da geração e pontuação desses lotes.
- Candidatos fixos: original congelado, mediana anterior, regra de divergência documental, HGB sem documento, regressão logística com documento e HGB com documento.
- HGB usa 150 iterações, taxa0,07, até15 folhas, pelo menos40 exemplos por folha e regularizaçãoL2=1, sem early stopping ou busca após o teste. A referência sem contexto é treinada com os mesmos exemplos/rótulos e parâmetros.
- Seleção por F0,5 com recall mínimo de50% na validação. Critério experimental de sucesso no teste nominal: precisão>=60%, recall>=50% e AP/F1 superiores ao HGB sem contexto nas mesmas transações. Não se trata de meta de negócio validada.

## Resultado nominal pareado

O selecionado foi **HGB com contexto**. Nos cinco clientes reservados há 8179 transações e 331 adulterações. A precisão foi **90.56%**, o recall **49.24%**, o F1 **0.6380** e a AP **0.7409**. Houve 180 alertas, 163 verdadeiros positivos, 17 falsos positivos e 168 falsos negativos. O resultado **não atendeu** ao critério registrado.

O HGB sem contexto, treinado sobre os mesmos exemplos, teve precisão de 7.00%, recall de 35.95%, F1 de 0.1172 e AP de 0.0800. Esta é a comparação apropriada para examinar a contribuição da informação adicional. Comparar diretamente estes números aos24,13% da rodada anterior misturaria populações e contratos diferentes. Todos os candidatos aparecem em `teste_agregado.csv`, sem promover outro modelo após olhar o teste.

## Dependência da qualidade documental

Com documentos degradados nos mesmos lotes, a precisão do selecionado foi 31.69% e o recall 27.19%. O desafio remove aleatoriamente 40% dos documentos, adiciona erros em 20% dos restantes e ruído logarítmico 0,08 ao preço. Sem documentos, a precisão foi 50.00% e o recall 0.30%, com apenas 2 alertas e 1 verdadeiro positivo. A precisão de um conjunto tão pequeno de alertas não demonstra utilidade; 330 adulterações ficaram sem detecção. O modelo e o limiar não foram recalibrados nos desafios. Os cenários são pareados e não correspondem a quinze clientes independentes.

O controle negativo com cinquenta permutações do gabarito produziu AP média de 0.0412, comparada à prevalência de 4.05%. Ele não substitui auditoria de disponibilidade da informação ou validação real.

## Limites e conclusão

O ganho depende da hipótese de um documento prévio parcialmente confiável. Esse campo não existe no CSV original, e não se afirma que o detector antigo passou a funcionar melhor sem nova informação. A nova abordagem é supervisionada, com rótulos artificiais; os experimentos antigos continuam não supervisionados. A capacidade de generalizar além do gerador e de processos com esse tipo de reconciliação não foi demonstrada. Um pedido incorreto pode produzir alerta falso e um documento ausente limita a detecção. Adulterações coordenadas do pedido e do lançamento não foram simuladas; padrões e faixas de multiplicadores do gerador podem favorecer o classificador.

O alvo completo de precisão>=60% e recall>=50% não foi atingido, pois o recall foi 49.24%. A distância para o piso de recall foi 0.76 ponto percentual. Não se alterou o limiar após observar essa diferença. O ganho de precisão, AP e F1 frente ao controle é apresentado separadamente da aprovação pelo critério completo.

Não foram acrescentados indicadores de autorização derivados do rótulo, alteradas as classes dos testes antigos ou filtrados os casos difíceis para elevar métricas. Uma adoção real exige confirmar disponibilidade, integridade e semântica do documento, obter rótulos confiáveis e realizar avaliação externa. O sistema sinaliza inconsistência, sem comprovar fraude.

## Inferência e reprodução

Usar as dependências de `ml/requirements.txt`, Python3.11/3.12. `python -m ml.context_run` reproduz o experimento. O contrato contém os sete campos originais e `id_documento`, `data_documento`, `quantidade_documento`, `preco_unitario_documento`. Os quatro campos documentais devem estar todos preenchidos ou todos ausentes. Documento preenchido deve ter data anterior ao lançamento.

`python -m ml.context_run --score-only --history HIST.csv --batch LOTE.csv --output SAIDA.csv` não abre gabarito nem recalibra o modelo. Os dados gerados, modelos e hashes ficam em `ml/artifacts/contexto/rodada_03` e `ml/reports/contexto/rodada_03`. O notebook08 e o consolidado apenas leem os resultados.


,modelo,arquivo,elegivel,average_precision,roc_auc,precision,recall,f1,precision_top_5pct,recall_top_5pct,...,taxa_alerta,tn,fp,fn,tp,limiar,n,anomalias,prevalencia,f0_5
0,HGB com contexto,candidato_5.joblib,True,0.7658,0.9534,0.9640,0.5323,0.6859,0.5961,0.7562,...,0.0218,4885,4,94,107,0.6721,5090,201,0.0395,0.8295
1,Regra documental,candidato_2.joblib,True,0.7084,0.8428,0.8039,0.6119,0.6949,0.5882,0.7463,...,0.0301,4859,30,78,123,0.1913,5090,201,0.0395,0.7565
2,Logistica com contexto,candidato_4.joblib,True,0.7370,0.9532,0.8067,0.6020,0.6895,0.5961,0.7562,...,0.0295,4860,29,80,121,0.1896,5090,201,0.0395,0.7553
3,Original congelado,candidato_0.joblib,False,0.0832,0.6206,0.1009,0.2836,0.1488,0.1294,0.1642,...,0.1110,4381,508,144,57,1.6812,5090,201,0.0395,0.1158
4,Mediana anterior,candidato_1.joblib,False,0.0708,0.6214,0.0967,0.1294,0.1106,0.0980,0.1244,...,0.0528,4646,243,175,26,2.7882,5090,201,0.0395,0.1018
5,HGB sem contexto,candidato_3.joblib,True,0.0849,0.6406,0.0721,0.5025,0.1261,0.1294,0.1642,...,0.2752,3589,1300,100,101,0.0370,5090,201,0.0395,0.0870


,cenario,modelo,selecionado,average_precision,roc_auc,precision,recall,f1,precision_top_5pct,recall_top_5pct,...,taxa_alerta,tn,fp,fn,tp,limiar,n,anomalias,prevalencia,f0_5
0,documentos_degradados,HGB com contexto,True,0.3126,0.7736,0.3169,0.2719,0.2927,0.2567,0.3172,...,0.0347,7654,194,241,90,0.6721,8179,331,0.0405,0.3067
1,documentos_degradados,HGB sem contexto,False,0.0800,0.6043,0.0700,0.3595,0.1172,0.1247,0.1541,...,0.2077,6268,1580,212,119,0.0370,8179,331,0.0405,0.0835
2,documentos_degradados,Logistica com contexto,False,0.3101,0.7852,0.1828,0.3595,0.2424,0.2616,0.3233,...,0.0796,7316,532,212,119,0.1896,8179,331,0.0405,0.2027
3,documentos_degradados,Mediana anterior,False,0.0701,0.6341,0.1061,0.1269,0.1155,0.1027,0.1269,...,0.0484,7494,354,289,42,2.7882,8179,331,0.0405,0.1097
4,documentos_degradados,Original congelado,False,0.0839,0.6338,0.1034,0.2659,0.1489,0.1271,0.1571,...,0.1040,7085,763,243,88,1.6812,8179,331,0.0405,0.1178
5,documentos_degradados,Regra documental,False,0.2824,0.5862,0.1748,0.3776,0.2390,0.2567,0.3172,...,0.0874,7258,590,206,125,0.1913,8179,331,0.0405,0.1959
6,nominal,HGB com contexto,True,0.7409,0.9566,0.9056,0.4924,0.6380,0.5917,0.7311,...,0.0220,7831,17,168,163,0.6721,8179,331,0.0405,0.7755
7,nominal,HGB sem contexto,False,0.0800,0.6043,0.0700,0.3595,0.1172,0.1247,0.1541,...,0.2077,6268,1580,212,119,0.0370,8179,331,0.0405,0.0835
8,nominal,Logistica com contexto,False,0.7121,0.9526,0.7614,0.6073,0.6756,0.5795,0.7160,...,0.0323,7785,63,130,201,0.1896,8179,331,0.0405,0.7246
9,nominal,Mediana anterior,False,0.0701,0.6341,0.1061,0.1269,0.1155,0.1027,0.1269,...,0.0484,7494,354,289,42,2.7882,8179,331,0.0405,0.1097


,cenario,segmento,n,anomalias,tp,fp,precision,recall,fpr
0,nominal,Lote completo,8179,331,163,17,0.9056,0.4924,0.0022
1,nominal,Com documento,6947,277,163,17,0.9056,0.5884,0.0025
2,nominal,Sem documento,1232,54,0,0,0.0000,0.0000,0.0000
3,nominal,Adulterações sutis,169,169,36,0,1.0000,0.2130,NaN
4,nominal,Adulterações amplas,162,162,127,0,1.0000,0.7840,NaN
5,nominal,Normais com pico legítimo,154,0,0,1,0.0000,NaN,0.0065
6,nominal,Normais em regime novo,1022,0,0,3,0.0000,NaN,0.0029
7,nominal,Normais com erro documental,114,0,0,16,0.0000,NaN,0.1404
8,documentos_degradados,Lote completo,8179,331,90,194,0.3169,0.2719,0.0247
9,documentos_degradados,Com documento,4155,164,89,194,0.3145,0.5427,0.0486


<Figure size 1100x583 with 1 Axes>

<Figure size 1100x550 with 1 Axes>

<Figure size 1100x550 with 1 Axes>